Q1的任务要求如下：
比较三种运输方案：

a. 仅使用太空电梯系统

b. 仅使用传统火箭

c. 两者结合

安装必要的依赖包

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 配置matplotlib中文显示和负号显示，这里我就直接用英文了一劳永逸
plt.rcParams['font.family'] = 'DejaVu Sans'
# plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
# plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

下面开始考虑第一小问a，仅使用太空电梯系统

### 赖特定律（Wright's Law）成本模型

赖特定律也称为学习曲线理论，表明随着累积生产量的增加，单位成本会按幂律下降。

**公式**: C(x) = C₁ · x^(-b)

**参数说明**:
- C(x): 累积载荷量为x时的单位成本 ($/kg)
- C₁: 初始基准成本 ($/kg)
- x: 累积发射总载荷量 (kg)
- b: 学习率参数 (通常在0.1-0.3之间)

In [ ]:
"""
赖特定律成本模型实现(基于年份)

关键改进：
- 学习曲线效应基于累积年份，而不是累积载荷量
- 技术改进来自发射经验积累，不是简单的载荷线性关系
"""

def wright_law_cost_annual_R(cumulative_years, C1, b):
    """
    基于赖特定律计算单位成本
    
    参数:
        cumulative_years: 累积发射年份
        C1: 初始基准成本 ($/kg) - 第一次发射的单位成本
        b: 学习率参数
    
    返回:
        单位成本 ($/kg)
    """
    if cumulative_years <= 0:
        return C1
    return C1 * (((5*cumulative_years+10)/10) ** (-b))

def wright_law_cost_annual_H(cumulative_years, C1, b):
    """
    基于赖特定律计算单位成本
    
    参数:
        cumulative_years: 累积发射年份
        C1: 初始基准成本 ($/kg) - 第一次发射的单位成本
        b: 学习率参数
    
    返回:
        单位成本 ($/kg)
    """
    if cumulative_years <= 0:
        return C1
    return C1 * (((cumulative_years+10)/10) ** (-b))


def calculate_total_cost_with_learning_R(total_payload_kg, payload_per_year_kg, C1, b):
    """
    计算考虑学习曲线效应的总成本
    
    参数:
        total_payload_kg: 总载荷量 (kg)
        payload_per_year_kg: 每年发射的载荷量 (kg)，例如150吨 = 150,000 kg
        C1: 初始基准成本 ($/kg)
        b: 学习率参数
    
    返回:
        total_cost: 总成本 ($)
        avg_cost: 平均单位成本 ($/kg)
        cost_history: 成本历史记录 [(发射次数, 累积载荷kg, 单位成本$/kg, 该年发射总成本$)]
        total_years: 总发射年数
    """
    # 计算所需的总发射年数
    total_years = int(np.ceil(total_payload_kg / payload_per_year_kg))
    
    total_cost = 0
    cumulative_payload = 0
    cost_history = []
    
    # 逐年发射计算
    for year_num in range(1, total_years + 1):
        # 计算这年发射的载荷（最后一次可能不足）
        if year_num < total_years:
            current_payload = payload_per_year_kg
        else:
            current_payload = total_payload_kg - cumulative_payload
        
        # 基于累积发射年数计算当前的单位成本
        unit_cost = wright_law_cost_annual_R(year_num, C1, b)
        
        # 计算本年度发射的总成本
        year_cost = unit_cost * current_payload
        total_cost += year_cost
        
        # 更新累积载荷
        cumulative_payload += current_payload
        
        # 记录历史（每隔一定次数记录，避免数据过多）
        cost_history.append((year_num, cumulative_payload, unit_cost, year_cost))
    
    avg_cost = total_cost / total_payload_kg
    return total_cost, avg_cost, cost_history, total_years

def calculate_total_cost_with_learning_H(total_payload_kg, payload_per_year_kg, C1, b):
    """
    计算考虑学习曲线效应的总成本
    
    参数:
        total_payload_kg: 总载荷量 (kg)
        payload_per_year_kg: 每年发射的载荷量 (kg)，例如150吨 = 150,000 kg
        C1: 初始基准成本 ($/kg)
        b: 学习率参数
    
    返回:
        total_cost: 总成本 ($)
        avg_cost: 平均单位成本 ($/kg)
        cost_history: 成本历史记录 [(发射次数, 累积载荷kg, 单位成本$/kg, 该年发射总成本$)]
        total_years: 总发射年数
    """
    # 计算所需的总发射年数
    total_years = int(np.ceil(total_payload_kg / payload_per_year_kg))
    
    total_cost = 0
    cumulative_payload = 0
    cost_history = []
    
    # 逐年发射计算
    for year_num in range(1, total_years + 1):
        # 计算这年发射的载荷（最后一次可能不足）
        if year_num < total_years:
            current_payload = payload_per_year_kg
        else:
            current_payload = total_payload_kg - cumulative_payload
        
        # 基于累积发射年数计算当前的单位成本
        unit_cost = wright_law_cost_annual_R(year_num, C1, b)
        
        # 计算本年度发射的总成本
        year_cost = unit_cost * current_payload
        total_cost += year_cost
        
        # 更新累积载荷
        cumulative_payload += current_payload
        
        # 记录历史（每隔一定次数记录，避免数据过多）
        cost_history.append((year_num, cumulative_payload, unit_cost, year_cost))
    
    avg_cost = total_cost / total_payload_kg
    return total_cost, avg_cost, cost_history, total_years

# 测试赖特定律模型
print("=== 赖特定律成本模型测试（基于年份）===\n")

# 参数设置
C1_rocket = 300  # 火箭初始成本 300$/kg

# 火箭基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
capacity_per_rocket = 150  # 每个火箭运输能力：100-150,这里取最大
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 3 # 每个基地每天发送频率
rocket_cost_per_kg = 300 # 火箭运输成本300-500美刀/kg，这里取300

# 火箭运输参数
payload_per_year = 365 * num_launch_site * num_launch_fre_per_day * capacity_per_rocket  # 每年发射吨数
payload_per_year_kg = payload_per_year * 1000  # 转换为kg
b_values = [0.15, 0.20, 0.30]  # 不同的学习率参数

# 测试案例：发射不同年数后的成本
test_years = [1, 10, 50, 150, 200]

print(f"每年发射载荷: {payload_per_year} 吨\n")

for b in b_values:
    print(f"学习率参数 b = {b} (每发射年份数翻倍，成本降至 {100*(2**(-b)):.1f}%)")
    print("-" * 80)
    print(f"{'发射年数':<12} {'累积载荷(吨)':<18} {'单位成本($/kg)':<20} {'成本下降':<15}")
    print("-" * 80)
    for years in test_years:
        cumulative_tons = years * payload_per_year
        unit_cost = wright_law_cost_annual_R(years, C1_rocket, b)
        reduction = (1 - unit_cost/C1_rocket) * 100
        print(f"{years:<12} {cumulative_tons:<18,} ${unit_cost:<19.2f} {reduction:.1f}%")
    print()


In [ ]:
"""
可视化赖特定律成本曲线
"""

# 生成累积载荷数据 200年
cumulative_years = np.logspace(0, np.log10(200), 1000)  # 200年
# 参数设置
C1_rocket = 300  # 初始成本 300美刀/kg
b_values = [0.15, 0.20, 0.30]

# 创建图表
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 左图：对数坐标
for b in b_values:
    costs = [wright_law_cost_annual_R(x, C1_rocket, b) for x in cumulative_years]
    ax1.loglog(cumulative_years, costs, linewidth=2, 
               label=f'b = {b} (reduction: {100*(1-2**(-b)):.1f}% per doubling)')

ax1.set_xlabel('Cumulative years', fontsize=12)
ax1.set_ylabel('Unit Cost ($/kg)', fontsize=12)
ax1.set_title('Wright\'s Law Cost Curve (Log-Log Scale)', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# 右图：线性坐标
cumulative_years_linear = np.linspace(1, 200, 1000)
for b in b_values:
    costs = [wright_law_cost_annual_R(x, C1_rocket, b) for x in cumulative_years_linear]
    ax2.plot(cumulative_years_linear, costs, linewidth=2, 
             label=f'b = {b}')

ax2.set_xlabel('Cumulative years', fontsize=12)
ax2.set_ylabel('Unit Cost ($/kg)', fontsize=12)
ax2.set_title('Wright\'s Law Cost Curve (Linear Scale, 0-200 years)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

print("图表说明：")
print("- 左图使用对数坐标，展示整个范围内的成本下降趋势")
print("- 右图使用线性坐标，聚焦在0-200年范围内的成本变化")
print("- 学习率参数b越大，成本下降越快")

In [ ]:
"""
计算仅使用太空电梯系统运输1亿吨物资所需的时间及金钱
(静态计算)
"""
# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
annual_capacity_per_harbor = 179_000  # 每个银河港口年运输能力：17.9万吨
num_harbors = 3  # 银河港口数量
habor_cost_per_kg = 100 # 太空电梯运输成本10-100美刀/kg，这里取100
    
print("=== 太空电梯系统运输能力计算 ===")
print(f"总物资需求: {total_materials:,} 吨")
print(f"每个银河港口年运输能力: {annual_capacity_per_harbor:,} 吨")
print(f"银河港口数量: {num_harbors}个")
    
# 计算总年运输能力
total_annual_capacity = annual_capacity_per_harbor * num_harbors
print(f"太空电梯系统总年运输能力: {total_annual_capacity:,} 吨")

# 计算总运输耗资
total_anual_cost = total_materials * 1000 * habor_cost_per_kg
print(f"太空电梯系统总运输耗资: {total_anual_cost:,} 美元")
    
# 计算所需年数
years_needed = total_materials / total_annual_capacity
print(f"理论所需年数: {years_needed:.2f} 年")
    
# 分解为年月日
full_years = int(years_needed)
months = int((years_needed - full_years) * 12)
days = int(((years_needed - full_years) * 12 - months) * 30.44)  # 平均每月30.44天
    
print(f"运输时间分解: {full_years} 年 {months} 月 {days} 天")
print(f"约合 {years_needed*12:.1f} 个月")
print(f"约合 {years_needed*365:.0f} 天")


    

In [ ]:
"""
使用赖特定律计算太空电梯运输的动态成本
"""

# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
total_materials_kg = total_materials * 1000  # 转换为kg


# 电梯基础参数
annual_capacity_per_harbor = 179_000  # 每个银河港口年运输能力：17.9万吨
num_harbors = 3  # 银河港口数量
habor_cost_per_kg = 100 # 太空电梯运输成本10-100美刀/kg，这里取100

# 电梯运输参数
payload_per_year = annual_capacity_per_harbor * num_harbors  # 每年运输吨数
payload_per_year_kg = payload_per_year * 1000  # 转换为kg

# 赖特定律参数
C1_harbor = 100  # 初始基准成本 ($/kg)
b_harbor = 0.15  # 学习率参数

print("=== 基于赖特定律的太空电梯运输成本计算 ===\n")
print(f"总物资需求: {total_materials:,} 吨 ({total_materials_kg:,} kg)")
print(f"每年运输载荷: {payload_per_year} 吨")
print(f"初始单位成本 C₁: ${C1_harbor}/kg")
print(f"学习率参数 b: {b_harbor}")
print(f"太空电梯系统总运输耗资: {total_anual_cost:,} (约 ${total_anual_cost/1e9:.2f} billion美元)")

# 计算总成本（基于发射次数）
total_cost, avg_cost, cost_history, total_years = calculate_total_cost_with_learning_H(
    total_materials_kg, payload_per_year_kg, C1_harbor, b_harbor
)
    
# 计算所需年数
years_needed = total_materials / total_annual_capacity
print(f"理论所需年数: {years_needed:.2f} 年")


print("-" * 70)
print(f"总运输年数: {total_years:,} 年")
print(f"总运输成本: ${total_cost:,.2f} (约 ${total_cost/1e9:.2f} billion美元)")
print(f"平均单位成本: ${avg_cost:.2f}/kg")
print(f"初始单位成本: ${C1_harbor}/kg")
print(f"成本节省: {100*(1 - avg_cost/C1_harbor):.2f}% (相比固定成本)")

# 显示关键年份的成本变化
print("\n关键年份成本变化：")
print("-" * 70)
print(f"{'年份':<12} {'累积载荷(吨)':<18} {'单位成本($/kg)':<20}")
print("-" * 70)
milestones = [1, 10, 100, 1000, 10000, 100000, total_years]
for milestone in milestones:
    if milestone <= total_years:
        unit_cost = wright_law_cost_annual_H(milestone, C1_harbor, b_harbor)
        cumulative_tons = min(milestone * payload_per_year, total_materials)
        print(f"{milestone:<12,} {cumulative_tons:<18,} ${unit_cost:<19.2f}")

print("-" * 70)

# 对比不同学习率下的成本
print("\n不同学习率参数下的总成本对比：")
print("-" * 70)
for b in [0.10, 0.15, 0.20, 0.25, 0.30]:
    cost, avg, _, launches = calculate_total_cost_with_learning_H(
        total_materials_kg, payload_per_year_kg, C1_harbor, b
    )
    print(f"b = {b:.2f}: 年数{launches:,}年 | 总成本 ${cost/1e9:.2f}billion美元 | 平均成本 ${avg:.2f}/kg | 节省 {100*(1-avg/C1_harbor):.1f}%")


下面开始考虑第二小问b，仅使用火箭运输

In [ ]:
"""
计算仅使用火箭系统运输1亿吨物资所需的时间及金钱
(静态计算)
"""
# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
capacity_per_rocket = 150  # 每个火箭运输能力：100-150,这里取最大
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 1 # 每个基地每天发送频率

rocket_cost_per_kg = 300 # 火箭运输成本300-500美刀/kg，这里取300
    
print("=== 火箭系统运输能力计算 ===")
print(f"总物资需求: {total_materials:,} 吨")
print(f"每个火箭运输能力: {capacity_per_rocket:,} 吨")
print(f"火箭基地数量: {num_launch_site}个")
print(f"每个基地每天发送频率: {num_launch_fre_per_day}次")
    
# 计算总年运输能力
total_annual_capacity = capacity_per_rocket * num_launch_site * num_launch_fre_per_day * 365
print(f"火箭系统总年运输能力: {total_annual_capacity:,} 吨")

# 计算总运输耗资
total_anual_cost = total_materials * 1000 * rocket_cost_per_kg
print(f"火箭系统总运输耗资: {total_anual_cost:,} 美元")
    
# 计算所需年数
years_needed = total_materials / total_annual_capacity
print(f"理论所需年数: {years_needed:.2f} 年")
    
# 分解为年月日
    
print(f"运输时间分解: {full_years} 年 {months} 月 {days} 天")
print(f"约合 {years_needed*12:.1f} 个月")
print(f"约合 {years_needed*365:.0f} 天")


print(f"约合 {years_needed*365:.0f} 天")

In [ ]:
"""
使用赖特定律计算火箭运输的动态成本
"""

# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
total_materials_kg = total_materials * 1000  # 转换为kg


# 火箭基础参数
capacity_per_rocket = 150  # 每个火箭运输能力：100-150,这里取最大
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 3 # 每个基地每天发送频率
rocket_cost_per_kg = 300 # 火箭运输成本300-500美刀/kg，这里取300

# 火箭运输参数
payload_per_year = 365 * num_launch_site * num_launch_fre_per_day * capacity_per_rocket  # 每年发射吨数
payload_per_year_kg = payload_per_year * 1000  # 转换为kg

# 赖特定律参数
C1_rocket = 300  # 初始基准成本 ($/kg)
b_rocket = 0.20  # 学习率参数

print("=== 基于赖特定律的火箭运输成本计算===\n")
print(f"总物资需求: {total_materials:,} 吨 ({total_materials_kg:,} kg)")
print(f"每年火箭发射载荷: {payload_per_year} 吨")
print(f"初始单位成本 C₁: ${C1_rocket}/kg")
print(f"学习率参数 b: {b_rocket}")
print(f"火箭系统总运输耗资: {total_anual_cost:,} (约 ${total_anual_cost/1e9:.2f} billion美元)")

# 计算总成本（基于发射次数）
total_cost, avg_cost, cost_history, total_years = calculate_total_cost_with_learning_R(
    total_materials_kg, payload_per_year_kg, C1_rocket, b_rocket
)

# 计算所需年数
years_needed = total_materials / total_annual_capacity
print(f"理论所需年数: {years_needed:.2f} 年")

print("-" * 70)
print(f"总发射年数: {total_years:,} 年")
print(f"总运输成本: ${total_cost:,.2f} (约 ${total_cost/1e9:.2f} billion美元)")
print(f"平均单位成本: ${avg_cost:.2f}/kg")
print(f"初始单位成本: ${C1_rocket}/kg")
print(f"成本节省: {100*(1 - avg_cost/C1_rocket):.2f}% (相比固定成本)")

# 显示关键发射节点的成本变化
print("\n关键年份成本变化：")
print("-" * 70)
print(f"{'发射次数':<12} {'累积载荷(吨)':<18} {'单位成本($/kg)':<20}")
print("-" * 70)
milestones = [1, 10, 100, 1000, 10000, 100000, total_years]
for milestone in milestones:
    if milestone <= total_years:
        unit_cost = wright_law_cost_annual_R(milestone, C1_rocket, b_rocket)
        cumulative_tons = min(milestone * payload_per_year, total_materials)
        print(f"{milestone:<12,} {cumulative_tons:<18,} ${unit_cost:<19.2f}")

print("-" * 70)

# 对比不同学习率下的成本
print("\n不同学习率参数下的总成本对比：")
print("-" * 70)
for b in [0.10, 0.15, 0.20, 0.25, 0.30]:
    cost, avg, _, launches = calculate_total_cost_with_learning_R(
        total_materials_kg, payload_per_year_kg, C1_rocket, b
    )
    print(f"b = {b:.2f}: 年数{launches:,}年 | 总成本 ${cost/1e9:.2f}billion美元 | 平均成本 ${avg:.2f}/kg | 节省 {100*(1-avg/C1_rocket):.1f}%")


In [ ]:
"""
可视化运输1亿吨物资过程中的成本演变
"""

# 提取成本历史数据 
# cost_history 格式: [(运输年数, 累积载荷kg, 单位成本$/kg, 本年发射总成本$)]
years_numbers = [x[0] for x in cost_history]  # 运输年数
cumulative_payload_tons = [x[1] / 1000 for x in cost_history]  # 转换为吨
unit_costs = [x[2] for x in cost_history]  # 单位成本

# 计算累积总成本
cumulative_costs = []
cumulative_cost = 0
for launch_num, cum_payload_kg, unit_cost, launch_cost in cost_history:
    cumulative_cost += launch_cost
    cumulative_costs.append(cumulative_cost / 1e9)  # 转换为billion美元

# 创建图表
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 子图1: 单位成本变化
ax1 = axes[0, 0]
ax1.plot(cumulative_payload_tons, unit_costs, linewidth=2, color='#2E86AB')
ax1.axhline(y=C1_rocket, color='red', linestyle='--', linewidth=1.5, 
            label=f'Initial cost ${C1_rocket}/kg')
ax1.axhline(y=avg_cost, color='green', linestyle='--', linewidth=1.5, 
            label=f'Average cost ${avg_cost:.2f}/kg')
ax1.fill_between(cumulative_payload_tons, unit_costs, C1_rocket, alpha=0.3, color='#A23B72')
ax1.set_xlabel('Cumulative payload (tons)', fontsize=11)
ax1.set_ylabel('Unit Cost ($/kg)', fontsize=11)
ax1.set_title('Unit Transportation Cost vs Cumulative Payload', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

# 子图2: 累积总成本
ax2 = axes[0, 1]
ax2.plot(cumulative_payload_tons, cumulative_costs, linewidth=2, color='#F18F01')
# 计算固定成本下的累积成本
fixed_cumulative_costs = [x * C1_rocket for x in cumulative_payload_tons]
ax2.plot(cumulative_payload_tons, fixed_cumulative_costs, linewidth=2, 
         color='red', linestyle='--', alpha=0.7, label='Fixed cost')
ax2.fill_between(cumulative_payload_tons, cumulative_costs, fixed_cumulative_costs, 
                  alpha=0.3, color='#C73E1D')
ax2.set_xlabel('Cumulative payload (tons)', fontsize=11)
ax2.set_ylabel('Cumulative Total Cost (Billion USD)', fontsize=11)
ax2.set_title('Cumulative Total Cost Comparison (Wright\'s Law vs Fixed Cost)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

# 子图3: 成本节省百分比
ax3 = axes[1, 0]
savings_percent = [(C1_rocket - uc) / C1_rocket * 100 for uc in unit_costs]
ax3.plot(cumulative_payload_tons, savings_percent, linewidth=2, color='#06A77D')
ax3.fill_between(cumulative_payload_tons, 0, savings_percent, alpha=0.3, color='#06A77D')
ax3.set_xlabel('Cumulative payload (tons)', fontsize=11)
ax3.set_ylabel('Cost Saving Percentage (%)', fontsize=11)
ax3.set_title('Cost Saving Percentage Relative to Initial Cost', fontsize=13, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 子图4: 关键指标对比
ax4 = axes[1, 1]
ax4.axis('off')

metrics_text = f"""
Key Metrics Summary
{'='*60}

Total Materials Transported:        {total_materials:,} tons

Wright's Law Parameters:
  • Initial cost C₁:     ${C1_rocket}/kg
  • Learning rate b:     {b_rocket}
  • Cost per doubling:   {100*(2**(-b_rocket)):.2f}%

Cost Analysis:
  • Total transport cost:      ${total_cost/1e9:.2f} billion USD
  • Average unit cost:         ${avg_cost:.2f}/kg
  • Final unit cost:           ${unit_costs[-1]:.2f}/kg

Comparison with Fixed Cost Scenario:
  • Fixed cost total:          ${(C1_rocket * total_materials_kg)/1e9:.2f} billion USD
  • Cost saving:               ${(C1_rocket * total_materials_kg - total_cost)/1e9:.2f} billion USD
  • Saving percentage:         {100*(1 - avg_cost/C1_rocket):.2f}%

Learning Curve Effect:
  • Starting cost:             ${C1_rocket}/kg
  • Ending cost:               ${unit_costs[-1]:.2f}/kg
  • Cost reduction:            {100*(C1_rocket - unit_costs[-1])/C1_rocket:.2f}%
"""

ax4.text(0.1, 0.95, metrics_text, transform=ax4.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

print("\n图表说明：")
print("• 左上：展示单位成本如何随累积运输量递减")
print("• 右上：对比赖特定律与固定成本方案的累积总成本差异")
print("• 左下：显示相对初始成本的节省百分比")
print("• 右下：汇总关键指标和对比数据")

下面开始考虑第三小问c，这两种方法的结合

首先考虑以最短时间完成建设任务，这里假设电梯和火箭都以最大限度运行

In [ ]:
"""
计算以最短时间完成建设任务所需的时间
"""
# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
annual_capacity_per_harbor = 179_000  # 每个银河港口年运输能力：17.9万吨
num_harbors = 3  # 银河港口数量

capacity_per_rocket = 150  # 每个火箭运输能力：100-150
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 3 # 每个基地每天发送频率


# 计算火箭总年运输能力
rocket_total_annual_capacity = capacity_per_rocket * num_launch_site * num_launch_fre_per_day * 365
print(f"火箭系统总年运输能力: {rocket_total_annual_capacity:,} 吨")

# 计算电梯总年运输能力
harbor_total_annual_capacity = annual_capacity_per_harbor * num_harbors
print(f"太空电梯系统总年运输能力: {harbor_total_annual_capacity:,} 吨")

total_annual_capacity = rocket_total_annual_capacity + harbor_total_annual_capacity

# 计算最短所需年数
years_needed = total_materials / total_annual_capacity
print(f"理论所需年数: {years_needed:.2f} 年")
    
# 分解为年月日
full_years = int(years_needed)
months = int((years_needed - full_years) * 12)
days = int(((years_needed - full_years) * 12 - months) * 30.44)  # 平均每月30.44天
    
print(f"运输时间分解: {full_years} 年 {months} 月 {days} 天")
print(f"约合 {years_needed*12:.1f} 个月")
print(f"约合 {years_needed*365:.0f} 天")

最省钱的只需考虑电梯即可

最后综合考虑时间以及金钱因素

## 多目标优化模型：时间-成本权衡分析

构建一个带权重的优化模型，在时间效率和成本效率之间寻找最优平衡点。

**决策变量**:
- α: 太空电梯运输的物资比例 (0 ≤ α ≤ 1)
- (1-α): 火箭运输的物资比例

**目标函数**:
$$\min Z = w_1 \cdot \frac{T(α)}{T_{min}} + w_2 \cdot \frac{C(α)}{C_{min}}$$

其中:
- T(α): 混合方案的总时间
- C(α): 混合方案的总成本（考虑赖特定律）
- T_min: 最短可能时间（两者全开）
- C_min: 最低可能成本（仅用最便宜的）
- w₁ + w₂ = 1（权重归一化）
- w₁ 时间权重
- w₂ 金钱权重


In [ ]:
"""
多目标优化模型实现（考虑赖特定律）
"""

from scipy.optimize import minimize_scalar

# ==================== 参数配置 ====================

# 基础参数
total_materials = 100_000_000  # 总物资需求：1亿吨
total_materials_kg = total_materials * 1000

# 太空电梯参数
# 电梯基础参数
annual_capacity_per_harbor = 179_000  # 每个银河港口年运输能力：17.9万吨
num_harbors = 3  # 银河港口数量

# 电梯运输参数
harbor_annual_capacity = annual_capacity_per_harbor * num_harbors  # 每年运输吨数
harbor_cost_per_kg = 100  # 固定成本 100$/kg
C1_harbor = 100
b_harbor = 0.15  # 太空电梯学习率（较低，技术相对成熟）

# 火箭参数  
# 火箭基础参数
capacity_per_rocket = 150  # 每个火箭运输能力：100-150,这里取最大
num_launch_site = 10  # 火箭基地数量
num_launch_fre_per_day = 3 # 每个基地每天发送频率
rocket_cost_per_kg = 300 # 火箭运输成本300-500美刀/kg，这里取300

# 火箭运输参数
rocket_annual_capacity = 365 * num_launch_site * num_launch_fre_per_day * capacity_per_rocket  # 每年发射吨数
C1_rocket = 300  # 初始成本 300$/kg
b_rocket = 0.25  # 火箭学习率（较高，技术快速进步）

print("=" * 80)
print("多目标优化模型：时间-成本权衡分析".center(80))
print("=" * 80)


# ==================== 成本计算函数（考虑赖特定律） ====================

def calculate_cost_with_learning_annual_R(payload_kg, annual_capacity_tons, C1, b):
    """
    基于年份的赖特定律计算总成本
    
    参数:
        payload_kg: 需要运输的载荷 (kg)
        annual_capacity_tons: 年运输能力 (tons)
        C1: 初始单位成本 ($/kg)
        b: 学习率参数
    """
    if payload_kg == 0:
        return 0
    
    payload_tons = payload_kg / 1000
    years_needed = payload_tons / annual_capacity_tons
    
    # 逐年计算成本（考虑学习曲线）
    total_cost = 0
    cumulative_years = 0
    
    # 分年计算，每年应用当年的成本
    full_years = int(years_needed)
    for year in range(1, full_years + 1):
        unit_cost = wright_law_cost_annual_R(year, C1, b)
        yearly_payload_kg = annual_capacity_tons * 1000  # 转换为kg
        total_cost += unit_cost * yearly_payload_kg
    
    # 计算最后不足一年的部分
    remaining_years = years_needed - full_years
    if remaining_years > 0:
        unit_cost = wright_law_cost_annual_R(full_years + 1, C1, b)
        yearly_payload_kg = remaining_years * annual_capacity_tons * 1000
        total_cost += unit_cost * yearly_payload_kg
    
    avg_cost = total_cost / payload_kg
    return total_cost, avg_cost, years_needed

def calculate_cost_with_learning_annual_H(payload_kg, annual_capacity_tons, C1, b):
    """
    基于年份的赖特定律计算总成本
    
    参数:
        payload_kg: 需要运输的载荷 (kg)
        annual_capacity_tons: 年运输能力 (tons)
        C1: 初始单位成本 ($/kg)
        b: 学习率参数
    """
    if payload_kg == 0:
        return 0
    
    payload_tons = payload_kg / 1000
    years_needed = payload_tons / annual_capacity_tons
    
    # 逐年计算成本（考虑学习曲线）
    total_cost = 0
    cumulative_years = 0
    
    # 分年计算，每年应用当年的成本
    full_years = int(years_needed)
    for year in range(1, full_years + 1):
        unit_cost = wright_law_cost_annual_H(year, C1, b)
        yearly_payload_kg = annual_capacity_tons * 1000  # 转换为kg
        total_cost += unit_cost * yearly_payload_kg
    
    # 计算最后不足一年的部分
    remaining_years = years_needed - full_years
    if remaining_years > 0:
        unit_cost = wright_law_cost_annual_H(full_years + 1, C1, b)
        yearly_payload_kg = remaining_years * annual_capacity_tons * 1000
        total_cost += unit_cost * yearly_payload_kg
    
    avg_cost = total_cost / payload_kg
    return total_cost, avg_cost, years_needed


# ==================== 混合方案评估函数 ====================

def evaluate_hybrid_scheme(alpha, w1=0.5, w2=0.5, verbose=False):
    """
    评估混合方案的综合得分
    
    参数:
        alpha: 太空电梯运输比例 (0-1)
        w1: 时间权重
        w2: 成本权重
        verbose: 是否输出详细信息
    
    返回:
        objective: 目标函数值（越小越好）
        time_years: 所需时间（年）
        total_cost: 总成本（美元）
    """
    # 计算各自承担的运输量
    harbor_payload_kg = total_materials_kg * alpha
    rocket_payload_kg = total_materials_kg * (1 - alpha)
    
    # 计算太空电梯部分
    if alpha > 0:
        harbor_cost, harbor_avg_cost, harbor_years = calculate_cost_with_learning_annual_H(
            harbor_payload_kg, harbor_annual_capacity, C1_harbor, b_harbor
        )
    else:
        harbor_cost, harbor_avg_cost, harbor_years = 0, 0, 0
    
    # 计算火箭部分
    if alpha < 1:
        rocket_cost, rocket_avg_cost, rocket_years = calculate_cost_with_learning_annual_R(
            rocket_payload_kg, rocket_annual_capacity, C1_rocket, b_rocket
        )
    else:
        rocket_cost, rocket_avg_cost, rocket_years = 0, 0, 0
    
    # 总时间 = max(两者时间)，因为并行运输
    time_years = max(harbor_years, rocket_years)
    
    # 总成本 = 两者成本之和
    total_cost = harbor_cost + rocket_cost
    
    if verbose:
        print(f"\n方案分析 (α = {alpha:.3f}):")
        print(f"  太空电梯: {harbor_payload_kg/1e11:.2f}亿吨 -> {harbor_years:.2f}年, ${harbor_cost/1e9:.2f}billion")
        print(f"  火箭系统: {rocket_payload_kg/1e11:.2f}亿吨 -> {rocket_years:.2f}年, ${rocket_cost/1e9:.2f}billion")
        print(f"  总时间: {time_years:.2f}年, 总成本: ${total_cost/1e9:.2f}billion")
    
    return time_years, total_cost


# ==================== 计算基准值 ====================

print("\n计算基准值...")
print("-" * 80)

# 方案1: 仅太空电梯 (α=1)
time_harbor_only, cost_harbor_only = evaluate_hybrid_scheme(1.0, verbose=True)

# 方案2: 仅火箭 (α=0)
time_rocket_only, cost_rocket_only = evaluate_hybrid_scheme(0.0, verbose=True)

# 方案3: 均衡分配（最短时间）
# 为了使时间最短，需要让两者同时完成
# harbor_payload / harbor_capacity = rocket_payload / rocket_capacity
# alpha / harbor_capacity = (1-alpha) / rocket_capacity
# alpha * rocket_capacity = (1-alpha) * harbor_capacity
# alpha = harbor_capacity / (harbor_capacity + rocket_capacity)

alpha_balanced = harbor_annual_capacity / (harbor_annual_capacity + rocket_annual_capacity)
time_balanced, cost_balanced = evaluate_hybrid_scheme(alpha_balanced, verbose=True)

# 确定最小时间和最小成本
T_min = time_balanced
C_min = min(cost_harbor_only, cost_rocket_only)

print(f"\n基准值:")
print(f"  最短时间 T_min = {T_min:.2f} 年")
print(f"  最低成本 C_min = ${C_min/1e9:.2f} billion美元")
print("-" * 80)

In [ ]:
"""
优化求解：寻找不同权重下的最优方案
"""

def objective_function(alpha, w1, w2):
    """
    目标函数：加权归一化的时间和成本
    """
    time_years, total_cost = evaluate_hybrid_scheme(alpha)
    
    # 归一化
    time_normalized = time_years / T_min
    cost_normalized = total_cost / C_min
    
    # 加权求和
    objective = w1 * time_normalized + w2 * cost_normalized
    
    return objective


# ==================== 不同权重组合下的最优解 ====================

print("\n不同权重组合下的最优方案:")
print("=" * 80)
print(f"{'权重(w1,w2)':<20} {'最优α':<12} {'时间(年)':<15} {'成本(billion$)':<15} {'目标值':<12}")
print("-" * 80)

weight_combinations = [
    (1.0, 0.0),   # 完全重视时间
    (0.8, 0.2),   # 主要重视时间
    (0.6, 0.4),   # 偏向时间
    (0.5, 0.5),   # 均衡
    (0.4, 0.6),   # 偏向成本
    (0.2, 0.8),   # 主要重视成本
    (0.0, 1.0),   # 完全重视成本
]

optimal_solutions = []

for w1, w2 in weight_combinations:
    # 使用scipy优化求解
    result = minimize_scalar(
        lambda a: objective_function(a, w1, w2),
        bounds=(0, 1),
        method='bounded'
    )
    
    alpha_opt = result.x
    obj_value = result.fun
    
    # 计算该方案的时间和成本
    time_opt, cost_opt = evaluate_hybrid_scheme(alpha_opt)
    
    optimal_solutions.append({
        'w1': w1,
        'w2': w2,
        'alpha': alpha_opt,
        'time_years': time_opt,
        'cost_billion': cost_opt / 1e9,
        'objective': obj_value
    })
    
    print(f"({w1:.1f}, {w2:.1f}){'':<12} {alpha_opt:<12.4f} {time_opt:<15.2f} {cost_opt/1e9:<15.2f} {obj_value:<12.4f}")

print("=" * 80)
# 转换为DataFrame便于后续分析
df_solutions = pd.DataFrame(optimal_solutions)

In [ ]:
"""
重点分析：w1=0.5, w2=0.5 的均衡方案
"""

# 找到均衡权重下的最优解
balanced_solution = df_solutions[df_solutions['w1'] == 0.5].iloc[0]
alpha_balanced_opt = balanced_solution['alpha']

print("\n" + "=" * 80)
print("重点推荐方案：均衡权重 (w1=0.5, w2=0.5)".center(80))
print("=" * 80)

# 详细评估
harbor_payload_kg = total_materials_kg * alpha_balanced_opt
rocket_payload_kg = total_materials_kg * (1 - alpha_balanced_opt)

print(f"\n【最优运输分配】")
print(f"  太空电梯承担: {alpha_balanced_opt*100:.2f}% ({harbor_payload_kg/1e9:.2f} 万吨)")
print(f"  火箭系统承担: {(1-alpha_balanced_opt)*100:.2f}% ({rocket_payload_kg/1e9:.2f} 万吨)")

# 计算太空电梯部分
if alpha_balanced_opt > 0:
    harbor_cost, harbor_avg_cost, harbor_years = calculate_cost_with_learning_annual_H(
        harbor_payload_kg, harbor_annual_capacity, C1_harbor, b_harbor
    )
    print(f"\n【太空电梯详情】")
    print(f"  运输量: {harbor_payload_kg/1e9:.2f} 万吨")
    print(f"  所需时间: {harbor_years:.2f} 年 ({harbor_years*12:.1f} 个月)")
    print(f"  初始成本: ${C1_harbor}/kg")
    print(f"  平均成本: ${harbor_avg_cost:.2f}/kg (学习曲线后)")
    print(f"  总成本: ${harbor_cost/1e9:.2f} billion美元")

# 计算火箭部分
if alpha_balanced_opt < 1:
    rocket_cost, rocket_avg_cost, rocket_years = calculate_cost_with_learning_annual_R(
        rocket_payload_kg, rocket_annual_capacity, C1_rocket, b_rocket
    )
    print(f"\n【火箭系统详情】")
    print(f"  运输量: {rocket_payload_kg/1e9:.2f} 万吨")
    print(f"  所需时间: {rocket_years:.2f} 年 ({rocket_years*12:.1f} 个月)")
    print(f"  初始成本: ${C1_rocket}/kg")
    print(f"  平均成本: ${rocket_avg_cost:.2f}/kg (学习曲线后)")
    print(f"  总成本: ${rocket_cost/1e9:.2f} billion美元")

# 汇总
time_total = max(harbor_years, rocket_years)
cost_total = harbor_cost + rocket_cost

print(f"\n【方案总结】")
print(f"  总时间: {time_total:.2f} 年 (并行运输)")
print(f"  总成本: ${cost_total/1e9:.2f} billion美元")
print(f"  平均单位成本: ${cost_total/total_materials_kg:.2f}/kg")

# 与极端方案对比
print(f"\n【与极端方案对比】")
print(f"  vs 仅电梯方案:")
print(f"    时间节省: {time_harbor_only - time_total:.2f} 年 ({100*(time_harbor_only-time_total)/time_harbor_only:.1f}%)")
print(f"    成本增加: ${(cost_total - cost_harbor_only)/1e9:.2f} billion ({100*(cost_total-cost_harbor_only)/cost_harbor_only:.1f}%)")

print(f"  vs 仅火箭方案:")
print(f"    时间节省: {time_rocket_only - time_total:.2f} 年 ({100*(time_rocket_only-time_total)/time_rocket_only:.1f}%)")
print(f"    成本节省: ${(cost_rocket_only - cost_total)/1e9:.2f} billion ({100*(cost_rocket_only-cost_total)/cost_rocket_only:.1f}%)")

print(f"  vs 均衡分配方案 (最短时间):")
print(f"    时间差异: {time_total - time_balanced:.2f} 年")
print(f"    成本节省: ${(cost_balanced - cost_total)/1e9:.2f} billion ({100*(cost_balanced-cost_total)/cost_balanced:.1f}%)")

print("=" * 80)

In [ ]:
"""
可视化分析：帕累托前沿与权重影响
"""

fig = plt.figure(figsize=(18, 12))

# ==================== 子图1: 帕累托前沿（时间-成本空间） ====================
ax1 = plt.subplot(2, 3, 1)

# 绘制连续的帕累托前沿
alpha_range = np.linspace(0, 1, 101)
pareto_times = []
pareto_costs = []

for alpha in alpha_range:
    t, c = evaluate_hybrid_scheme(alpha)
    pareto_times.append(t)
    pareto_costs.append(c / 1e9)

ax1.plot(pareto_times, pareto_costs, 'b-', linewidth=2, alpha=0.5, label='Pareto Frontier')
ax1.scatter(df_solutions['time_years'], df_solutions['cost_billion'], 
           c=df_solutions['w1'], cmap='RdYlGn_r', s=200, edgecolors='black', linewidth=2,
           alpha=0.8, label='Optimal Solutions')

# 标注关键点
ax1.scatter([time_harbor_only], [cost_harbor_only/1e9], 
           marker='s', s=300, c='green', edgecolors='black', linewidth=2, 
           label='Harbor Only', zorder=5)
ax1.scatter([time_rocket_only], [cost_rocket_only/1e9], 
           marker='^', s=300, c='red', edgecolors='black', linewidth=2, 
           label='Rocket Only', zorder=5)
ax1.scatter([balanced_solution['time_years']], [balanced_solution['cost_billion']], 
           marker='*', s=500, c='gold', edgecolors='black', linewidth=2, 
           label='Balanced (w1=w2=0.5)', zorder=5)

ax1.set_xlabel('Time (years)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cost (Billion USD)', fontsize=12, fontweight='bold')
ax1.set_title('Pareto Frontier: Time-Cost Tradeoff', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9, loc='best')

# 添加colorbar
cbar1 = plt.colorbar(ax1.collections[1], ax=ax1)
cbar1.set_label('Time Weight (w1)', fontsize=10)

# ==================== 子图2: 权重 vs 最优α ====================
ax2 = plt.subplot(2, 3, 2)

ax2.plot(df_solutions['w1'], df_solutions['alpha'], 'bo-', linewidth=2, markersize=10)
ax2.axhline(y=alpha_balanced_opt, color='gold', linestyle='--', linewidth=2, 
            label=f'Balanced α = {alpha_balanced_opt:.3f}')
ax2.fill_between(df_solutions['w1'], 0, df_solutions['alpha'], alpha=0.3, color='blue')

ax2.set_xlabel('Time Weight (w1)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Optimal Harbor Ratio (α)', fontsize=12, fontweight='bold')
ax2.set_title('Impact of Weight on Optimal Allocation', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

# ==================== 子图3: 权重 vs 时间和成本 ====================
ax3 = plt.subplot(2, 3, 3)

ax3_twin = ax3.twinx()

line1 = ax3.plot(df_solutions['w1'], df_solutions['time_years'], 
                'b-o', linewidth=2, markersize=8, label='Time')
line2 = ax3_twin.plot(df_solutions['w1'], df_solutions['cost_billion'], 
                     'r-s', linewidth=2, markersize=8, label='Cost')

ax3.set_xlabel('Time Weight (w1)', fontsize=12, fontweight='bold')
ax3.set_ylabel('Time (years)', fontsize=12, fontweight='bold', color='blue')
ax3_twin.set_ylabel('Cost (Billion USD)', fontsize=12, fontweight='bold', color='red')
ax3.set_title('Time and Cost vs Weight', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 合并图例
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, fontsize=10, loc='upper left')

# ==================== 子图4: α vs 时间和成本 ====================
ax4 = plt.subplot(2, 3, 4)

ax4.plot(alpha_range, pareto_times, 'b-', linewidth=2, label='Time')
ax4_twin = ax4.twinx()
ax4_twin.plot(alpha_range, pareto_costs, 'r-', linewidth=2, label='Cost')

# 标注最优点
for _, sol in df_solutions.iterrows():
    if sol['w1'] in [0.0, 0.5, 1.0]:
        ax4.scatter([sol['alpha']], [sol['time_years']], s=100, zorder=5)
        ax4_twin.scatter([sol['alpha']], [sol['cost_billion']], s=100, zorder=5)

ax4.set_xlabel('Harbor Allocation Ratio (α)', fontsize=12, fontweight='bold')
ax4.set_ylabel('Time (years)', fontsize=12, fontweight='bold', color='blue')
ax4_twin.set_ylabel('Cost (Billion USD)', fontsize=12, fontweight='bold', color='red')
ax4.set_title('Time and Cost vs Allocation Ratio', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

lines4 = ax4.get_lines() + ax4_twin.get_lines()
labels4 = [l.get_label() for l in lines4]
ax4.legend(lines4, labels4, fontsize=10, loc='upper left')

# ==================== 子图5: 目标函数值 ====================
ax5 = plt.subplot(2, 3, 5)

colors = plt.cm.RdYlGn_r(df_solutions['w1'])
bars = ax5.bar(range(len(df_solutions)), df_solutions['objective'], 
               color=colors, edgecolor='black', linewidth=1.5, alpha=0.8)

# 标注w1值
for i, (idx, row) in enumerate(df_solutions.iterrows()):
    ax5.text(i, row['objective'] + 0.01, f"w1={row['w1']:.1f}", 
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax5.set_xlabel('Solution Index', fontsize=12, fontweight='bold')
ax5.set_ylabel('Objective Value', fontsize=12, fontweight='bold')
ax5.set_title('Objective Function Value Comparison', fontsize=14, fontweight='bold')
ax5.grid(True, alpha=0.3, axis='y')

# ==================== 子图6: 运输比例饼图（均衡方案） ====================
ax6 = plt.subplot(2, 3, 6)

sizes = [alpha_balanced_opt * 100, (1 - alpha_balanced_opt) * 100]
labels_pie = [f'Space Elevator\n{alpha_balanced_opt*100:.1f}%\n({harbor_payload_kg/1e9:.1f}M tons)',
              f'Rocket\n{(1-alpha_balanced_opt)*100:.1f}%\n({rocket_payload_kg/1e9:.1f}M tons)']
colors_pie = ['#3498db', '#e74c3c']
explode = (0.05, 0.05)

ax6.pie(sizes, explode=explode, labels=labels_pie, colors=colors_pie, autopct='',
        shadow=True, startangle=90, textprops={'fontsize': 11, 'fontweight': 'bold'})
ax6.set_title(f'Optimal Allocation (w1=w2=0.5)\nTotal Time: {balanced_solution["time_years"]:.1f}y, Cost: ${balanced_solution["cost_billion"]:.1f}B', 
             fontsize=13, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n图表说明:")
print("-" * 80)
print("1. Pareto Frontier: 展示时间-成本的权衡曲线和不同权重下的最优解")
print("2. Weight Impact: 权重如何影响太空电梯的最优分配比例")
print("3. Time & Cost vs Weight: 随着时间权重增加，时间缩短但成本上升")
print("4. Time & Cost vs Allocation: 展示完整的帕累托前沿")
print("5. Objective Values: 不同权重下的目标函数值对比")
print("6. Optimal Allocation: 均衡权重下的运输分配饼图")
print("-" * 80)

# 改进的多目标优化模型

## 考虑负荷率和基地启用决策的动态优化模型

下面将实现一个更精细的优化模型，考虑：
1. 太空电梯的**负荷率调整** ($\mu_t$)：每年可以调整运输强度
2. 火箭基地的**启用决策** ($y_{i,t}$)：动态选择启用哪些基地
3. 维护成本和固定运营成本
4. 赖特定律的学习效应

# 太空电梯系统成本效率模型

## 成本效率函数


$$C_{H}(t) = Cost_{\text{maint}}(\mu_t) + Cost_{\text{ops}}(\mu_t)$$


## 决策变量

- **$\mu_t$**：第 $t$ 年太空电梯的**负荷率**（Load Factor）
  - **取值范围**：$0 \leq \mu_t \leq 1$
  - **物理意义**：
    - $\mu_t = 1$：满负荷运载
    - $\mu_t = 0$：停运状态（仍需基础维护）

维修成本包含“固定基础维护”和“动态磨损维护”。

$$Cost_{maint}(\mu_t) = W_{\text{base}} + W_{\text{wear}}\cdot(\mu_t)^\gamma$$

- **$W_{\text{base}}$**: 固定维护费(如地球港防腐、人员工资)。即使 $\mu_t$ = 0 也要花费
- **$W_{\text{wear}}$**: 满负荷下的额外磨损维修费
- **$\gamma$**: 磨损系数，通常取2或3，表示越快越伤

实际运输成本

$$Cost_{\text{ops}}(\mu_t) = \mu_t \cdot Cap_{\text{max}} \cdot UnitCost_{\text{max}}$$

这里的$UnitCost_{\text{max}}$指的是在赖特定律中计算出的单位成本，但实际运输中因为负荷率导致实际载货量变少


# 火箭系统成本效率模型

## 成本效率函数


$$C_{R}(t) = \sum_{i=1}^{10} (365 \cdot n_{\text{i,t}} \cdot c_{\text{i}}(t) + y_{\text{i,t}} \cdot F_{\text{i}}) $$

## 决策变量

- $n_{\text{i,t}}$ : 启用基地数量 (0-10)，且具体包含每个基地一一对应的情况
- $y_{\text{i,t}}$ : 启用基地数量 (0-10)，且具体包含每个基地一一对应的情况
- $c_{\text{i}}(t)$ : 每个基地的每天发射成本，与$n_{\text{i,t}}$ 对应
- $F_{\text{i}}$ :每个基地启用后的每天固定运营成本，与$y_{\text{i,t}}$ 对应

In [ ]:
"""
改进的成本计算函数：考虑负荷率和基地启用决策
"""

# 相关参数定义

W_base_harbor = 10
W_wear_harbor = 10
gamma_harbor = 2
harbor_max_capacity_kg = harbor_annual_capacity * 1000

F_sites = np.array([1_500_000, 2_000_000, 3_500_000, 4_000_000, 2_500_000, 1_000_000, 3_000_000, 3_000_000, 1_000_000, 1_500_000])
num_launch_sites = 10
max_launches_per_site_per_day = 3



# ==================== 太空电梯改进成本模型 ====================

def calculate_harbor_cost_with_loadfactor(mu_schedule, total_years, C1_harbor, b_harbor):
    """
    计算考虑负荷率的太空电梯总成本
    
    参数:
        mu_schedule: 负荷率时间表 numpy array, shape=(total_years,), 每年的负荷率 [0,1]
        total_years: 总年数
        C1_harbor: 初始单位运输成本 ($/kg)
        b_harbor: 学习率参数
        
    返回:
        total_cost: 总成本 ($)
        total_payload_kg: 总运输量 (kg)
        cost_breakdown: 详细成本分解列表
    """
    total_cost = 0
    total_payload_kg = 0
    cost_breakdown = []
    
    for year in range(1, total_years + 1):
        mu_t = mu_schedule[year - 1]  # 第t年的负荷率
        
        # 1. 维护成本
        cost_maint = W_base_harbor + W_wear_harbor * (mu_t ** gamma_harbor)
        
        # 2. 实际运输成本（考虑赖特定律）
        actual_capacity_kg = mu_t * harbor_max_capacity_kg  # 实际年运输量
        unit_cost_t = wright_law_cost_annual_H(year, C1_harbor, b_harbor)
        cost_ops = actual_capacity_kg * unit_cost_t
        
        # 年度总成本
        year_cost = cost_maint + cost_ops
        total_cost += year_cost
        total_payload_kg += actual_capacity_kg
        
        cost_breakdown.append({
            'year': year,
            'load_factor': mu_t,
            'payload_kg': actual_capacity_kg,
            'unit_cost': unit_cost_t,
            'cost_maint': cost_maint,
            'cost_ops': cost_ops,
            'year_cost': year_cost
        })
    
    return total_cost, total_payload_kg, cost_breakdown


# ==================== 火箭改进成本模型 ====================

def calculate_rocket_cost_with_sites(site_schedule, launches_schedule, total_years, C1_rocket, b_rocket):
    """
    计算考虑基地启用决策的火箭总成本
    
    参数:
        site_schedule: 基地启用时间表 numpy array, shape=(total_years, num_launch_sites)
                      site_schedule[t, i] = 1 表示第t年启用第i个基地, 0表示不启用
        launches_schedule: 发射次数时间表 numpy array, shape=(total_years, num_launch_sites)
                          launches_schedule[t, i] 表示第t年第i个基地的每天发射次数 [0, max_launches_per_site_per_day]
        total_years: 总年数
        C1_rocket: 初始单位发射成本 ($/kg)
        b_rocket: 学习率参数
        
    返回:
        total_cost: 总成本 ($)
        total_payload_kg: 总运输量 (kg)
        cost_breakdown: 详细成本分解列表
    """
    total_cost = 0
    total_payload_kg = 0
    cost_breakdown = []
    
    for year in range(1, total_years + 1):
        year_idx = year - 1
        y_t = site_schedule[year_idx]  # 第t年各基地的启用状态
        n_t = launches_schedule[year_idx]  # 第t年各基地的发射次数
        
        # 1. 固定运营成本（只要基地启用就产生）
        cost_fixed = np.sum(y_t * F_sites)  # 年度固定成本
        
        # 2. 发射成本（考虑赖特定律）
        unit_cost_t = wright_law_cost_annual_R(year, C1_rocket, b_rocket)
        
        # 计算本年度每个基地的发射量
        year_payload_kg = 0
        cost_launches = 0
        for i in range(num_launch_sites):
            if y_t[i] > 0:  # 基地i启用
                site_annual_launches = n_t[i] * 365  # 年度发射次数
                site_payload_kg = site_annual_launches * capacity_per_rocket * 1000  # 转换为kg
                site_cost = site_payload_kg * unit_cost_t
                
                year_payload_kg += site_payload_kg
                cost_launches += site_cost
        
        # 年度总成本
        year_cost = cost_fixed + cost_launches
        total_cost += year_cost
        total_payload_kg += year_payload_kg
        
        cost_breakdown.append({
            'year': year,
            'sites_active': np.sum(y_t),
            'payload_kg': year_payload_kg,
            'unit_cost': unit_cost_t,
            'cost_fixed': cost_fixed,
            'cost_launches': cost_launches,
            'year_cost': year_cost
        })
    
    return total_cost, total_payload_kg, cost_breakdown


print("=" * 80)
print("改进成本模型已定义".center(80))
print("=" * 80)
print("\n模型特点:")
print("1. 太空电梯：考虑负荷率调整 μ_t ∈ [0,1]，包含固定维护和动态磨损成本")
print("2. 火箭系统：考虑基地启用决策 y_i,t ∈ {0,1}，包含固定运营和发射成本")
print("3. 赖特定律：单位运输成本随经验积累递减")

In [ ]:
"""
多目标优化模型实现：简化策略版本

策略假设：
1. 太空电梯：采用恒定负荷率运行（简化决策）
2. 火箭：优先启用成本最低的基地，按需增加基地数量

这是一个实用的启发式策略，便于快速求解
"""

def evaluate_hybrid_scheme_improved(alpha, mu_harbor, num_sites_rocket, T_max, w1=0.5, w2=0.5):
    """
    评估改进的混合方案
    
    参数:
        alpha: 太空电梯承担的运输比例 [0, 1]
        mu_harbor: 太空电梯的恒定负荷率 [0, 1]
        num_sites_rocket: 火箭启用的基地数量 [0, num_launch_sites]
        T_max: 最大允许时间（年）
        w1: 时间权重
        w2: 成本权重
        
    返回:
        is_feasible: 是否满足时间约束
        time_years: 所需时间（年）
        total_cost: 总成本（$）
        objective: 目标函数值
    """
    
    # 1. 计算太空电梯部分
    harbor_target_kg = total_materials_kg * alpha
    
    if harbor_target_kg > 0 and mu_harbor > 0:
        # 实际年运输能力
        harbor_actual_annual_capacity_kg = mu_harbor * harbor_max_capacity_kg
        harbor_years_needed = harbor_target_kg / harbor_actual_annual_capacity_kg
        
        # 构建负荷率时间表（恒定负荷率）
        harbor_total_years = int(np.ceil(harbor_years_needed))
        mu_schedule = np.full(harbor_total_years, mu_harbor)
        
        # 最后一年可能不需要满负荷
        if harbor_years_needed % 1 > 0:
            last_year_ratio = harbor_years_needed % 1
            mu_schedule[-1] = mu_harbor * last_year_ratio
        
        # 计算成本
        harbor_cost, harbor_actual_payload, _ = calculate_harbor_cost_with_loadfactor(
            mu_schedule, harbor_total_years, C1_harbor, b_harbor
        )
        harbor_time = harbor_years_needed
    else:
        harbor_cost = 0
        harbor_actual_payload = 0
        harbor_time = 0
    
    # 2. 计算火箭部分
    rocket_target_kg = total_materials_kg * (1 - alpha)
    
    if rocket_target_kg > 0 and num_sites_rocket > 0:
        # 选择成本最低的几个基地
        site_costs_sorted_idx = np.argsort(F_sites)[:int(num_sites_rocket)]
        
        # 计算火箭年运输能力
        rocket_annual_capacity_kg = (num_sites_rocket * max_launches_per_site_per_day * 
                                     365 * capacity_per_rocket * 1000)
        rocket_years_needed = rocket_target_kg / rocket_annual_capacity_kg
        
        # 构建基地启用和发射时间表
        rocket_total_years = int(np.ceil(rocket_years_needed))
        site_schedule = np.zeros((rocket_total_years, num_launch_sites))
        launches_schedule = np.zeros((rocket_total_years, num_launch_sites))
        
        # 启用选中的基地，满负荷运行
        for year_idx in range(rocket_total_years):
            for site_idx in site_costs_sorted_idx:
                site_schedule[year_idx, site_idx] = 1  # 启用基地
                # 最后一年可能不需要满负荷
                if year_idx == rocket_total_years - 1 and rocket_years_needed % 1 > 0:
                    last_year_ratio = rocket_years_needed % 1
                    launches_schedule[year_idx, site_idx] = max_launches_per_site_per_day * last_year_ratio
                else:
                    launches_schedule[year_idx, site_idx] = max_launches_per_site_per_day
        
        # 计算成本
        rocket_cost, rocket_actual_payload, _ = calculate_rocket_cost_with_sites(
            site_schedule, launches_schedule, rocket_total_years, C1_rocket, b_rocket
        )
        rocket_time = rocket_years_needed
    else:
        rocket_cost = 0
        rocket_actual_payload = 0
        rocket_time = 0
    
    # 3. 汇总结果
    time_years = max(harbor_time, rocket_time) if (harbor_time > 0 or rocket_time > 0) else 0
    total_cost = harbor_cost + rocket_cost
    
    # 检查可行性
    is_feasible = (time_years <= T_max) and (time_years > 0)
    
    # 计算目标函数（归一化）
    if is_feasible:
        # 使用简单的归一化
        time_normalized = time_years / T_max
        cost_normalized = total_cost / (C1_harbor * total_materials_kg)  # 以初始成本为基准
        objective = w1 * time_normalized + w2 * cost_normalized
    else:
        objective = 1e10  # 不可行解的惩罚
    
    return is_feasible, time_years, total_cost, objective


print("=" * 80)
print("简化策略优化模型已定义".center(80))
print("=" * 80)

In [ ]:
"""
网格搜索求解：寻找最优策略组合
"""

print("\n" + "=" * 80)
print("网格搜索优化求解".center(80))
print("=" * 80)

# 设置最大允许时间为传统方案的时间
T_max = 200  # 年

# 定义搜索空间
alpha_values = np.linspace(0, 1, 21)  # 太空电梯比例：0%, 5%, ..., 100%
mu_harbor_values = np.array([0.3, 0.5, 0.7, 0.9])  # 负荷率
num_sites_values = np.arange(0, num_launch_sites + 1)  # 基地数量：0-10

# 不同权重组合
weight_combinations = [
    (1.0, 0.0, "Time Priority"),
    (0.7, 0.3, "Time Focused"),
    (0.5, 0.5, "Balanced"),
    (0.3, 0.7, "Cost Focused"),
    (0.0, 1.0, "Cost Priority"),
]

results_summary = []

for w1, w2, label in weight_combinations:
    print(f"\n{'=' * 80}")
    print(f"权重组合: {label} (w1={w1}, w2={w2})".center(80))
    print('=' * 80)
    
    best_objective = float('inf')
    best_solution = None
    all_feasible_solutions = []
    
    # 网格搜索
    search_count = 0
    for alpha in alpha_values:
        for mu_h in mu_harbor_values:
            for n_sites in num_sites_values:
                # 跳过不合理的组合
                if alpha == 0 and mu_h > 0:  # 不用电梯就不需要负荷率
                    continue
                if alpha == 1 and n_sites > 0:  # 只用电梯就不需要火箭基地
                    continue
                if alpha > 0 and mu_h == 0:  # 用电梯但负荷率为0没意义
                    continue
                if alpha < 1 and n_sites == 0:  # 用火箭但没有基地没意义
                    continue
                
                search_count += 1
                
                # 评估方案
                is_feasible, time_years, total_cost, objective = evaluate_hybrid_scheme_improved(
                    alpha, mu_h, n_sites, T_max, w1, w2
                )
                
                if is_feasible:
                    all_feasible_solutions.append({
                        'alpha': alpha,
                        'mu_harbor': mu_h,
                        'num_sites': n_sites,
                        'time_years': time_years,
                        'total_cost': total_cost / 1e9,  # 转换为billion美元
                        'objective': objective
                    })
                    
                    if objective < best_objective:
                        best_objective = objective
                        best_solution = {
                            'alpha': alpha,
                            'mu_harbor': mu_h,
                            'num_sites': n_sites,
                            'time_years': time_years,
                            'total_cost': total_cost,
                            'objective': objective
                        }
    
    # 输出最优解
    if best_solution:
        print(f"\n搜索方案数: {search_count}, 可行解数: {len(all_feasible_solutions)}")
        print(f"\n最优方案:")
        print(f"  太空电梯比例 α = {best_solution['alpha']:.2%}")
        print(f"  太空电梯负荷率 μ = {best_solution['mu_harbor']:.2%}")
        print(f"  火箭基地数量 = {best_solution['num_sites']:.0f} 个")
        print(f"  总时间 = {best_solution['time_years']:.2f} 年")
        print(f"  总成本 = ${best_solution['total_cost']/1e9:.2f} billion美元")
        print(f"  目标函数值 = {best_solution['objective']:.4f}")
        
        results_summary.append({
            'label': label,
            'w1': w1,
            'w2': w2,
            **best_solution
        })
    else:
        print(f"\n未找到可行解！")

print("\n" + "=" * 80)
print("所有权重组合的最优解汇总".center(80))
print("=" * 80)
print(f"{'策略':<20} {'α':<8} {'μ':<8} {'基地':<8} {'时间(年)':<12} {'成本(billion$)':<15}")
print("-" * 80)
for result in results_summary:
    print(f"{result['label']:<20} {result['alpha']:<8.2%} {result['mu_harbor']:<8.2%} "
          f"{result['num_sites']:<8.0f} {result['time_years']:<12.2f} {result['total_cost']/1e9:<15.2f}")

# 转换为DataFrame
df_results = pd.DataFrame(results_summary)

In [ ]:
"""
详细分析均衡权重方案
"""

# 找到均衡权重的最优解
balanced_result = df_results[df_results['w1'] == 0.5].iloc[0]

print("\n" + "=" * 80)
print("均衡权重方案详细分析 (w1=0.5, w2=0.5)".center(80))
print("=" * 80)

alpha_opt = balanced_result['alpha']
mu_opt = balanced_result['mu_harbor']
sites_opt = int(balanced_result['num_sites'])

print(f"\n【最优决策变量】")
print(f"  太空电梯承担比例 α = {alpha_opt:.2%}")
print(f"  太空电梯负荷率 μ = {mu_opt:.2%}")
print(f"  火箭启用基地数 = {sites_opt} 个")

# 重新计算以获取详细信息
harbor_target_kg = total_materials_kg * alpha_opt
rocket_target_kg = total_materials_kg * (1 - alpha_opt)

print(f"\n【运输量分配】")
print(f"  总物资需求: {total_materials:,} 吨")
print(f"  太空电梯承担: {harbor_target_kg/1e6:,.2f} 万吨 ({alpha_opt:.1%})")
print(f"  火箭系统承担: {rocket_target_kg/1e6:,.2f} 万吨 ({(1-alpha_opt):.1%})")

# 太空电梯详细分析
if alpha_opt > 0:
    harbor_actual_annual_capacity_kg = mu_opt * harbor_max_capacity_kg
    harbor_years_needed = harbor_target_kg / harbor_actual_annual_capacity_kg
    harbor_total_years = int(np.ceil(harbor_years_needed))
    
    # 构建负荷率时间表
    mu_schedule = np.full(harbor_total_years, mu_opt)
    if harbor_years_needed % 1 > 0:
        last_year_ratio = harbor_years_needed % 1
        mu_schedule[-1] = mu_opt * last_year_ratio
    
    # 计算成本
    harbor_cost, harbor_actual_payload, harbor_breakdown = calculate_harbor_cost_with_loadfactor(
        mu_schedule, harbor_total_years, C1_harbor, b_harbor
    )
    
    print(f"\n【太空电梯详细信息】")
    print(f"  最大年运输能力: {harbor_max_capacity_kg/1e6:,.2f} 万吨")
    print(f"  实际年运输能力: {harbor_actual_annual_capacity_kg/1e6:,.2f} 万吨 (负荷率{mu_opt:.1%})")
    print(f"  所需时间: {harbor_years_needed:.2f} 年")
    print(f"  总成本: ${harbor_cost/1e9:.2f} billion美元")
    
    # 成本分解
    total_maint = sum([b['cost_maint'] for b in harbor_breakdown])
    total_ops = sum([b['cost_ops'] for b in harbor_breakdown])
    print(f"    - 维护成本: ${total_maint/1e9:.2f} billion ({total_maint/harbor_cost*100:.1f}%)")
    print(f"    - 运输成本: ${total_ops/1e9:.2f} billion ({total_ops/harbor_cost*100:.1f}%)")
    
    # 学习效应
    first_year_unit_cost = harbor_breakdown[0]['unit_cost']
    last_year_unit_cost = harbor_breakdown[-1]['unit_cost']
    print(f"  单位成本变化:")
    print(f"    - 初始: ${first_year_unit_cost:.2f}/kg")
    print(f"    - 最终: ${last_year_unit_cost:.2f}/kg")
    print(f"    - 降幅: {(1 - last_year_unit_cost/first_year_unit_cost)*100:.1f}%")

# 火箭详细分析
if (1 - alpha_opt) > 0:
    # 选择成本最低的基地
    site_costs_sorted_idx = np.argsort(F_sites)[:sites_opt]
    selected_sites_costs = F_sites[site_costs_sorted_idx]
    
    rocket_annual_capacity_kg = (sites_opt * max_launches_per_site_per_day * 
                                 365 * capacity_per_rocket * 1000)
    rocket_years_needed = rocket_target_kg / rocket_annual_capacity_kg
    rocket_total_years = int(np.ceil(rocket_years_needed))
    
    # 构建时间表
    site_schedule = np.zeros((rocket_total_years, num_launch_sites))
    launches_schedule = np.zeros((rocket_total_years, num_launch_sites))
    
    for year_idx in range(rocket_total_years):
        for site_idx in site_costs_sorted_idx:
            site_schedule[year_idx, site_idx] = 1
            if year_idx == rocket_total_years - 1 and rocket_years_needed % 1 > 0:
                last_year_ratio = rocket_years_needed % 1
                launches_schedule[year_idx, site_idx] = max_launches_per_site_per_day * last_year_ratio
            else:
                launches_schedule[year_idx, site_idx] = max_launches_per_site_per_day
    
    # 计算成本
    rocket_cost, rocket_actual_payload, rocket_breakdown = calculate_rocket_cost_with_sites(
        site_schedule, launches_schedule, rocket_total_years, C1_rocket, b_rocket
    )
    
    print(f"\n【火箭系统详细信息】")
    print(f"  启用基地数: {sites_opt} 个")
    print(f"  选中的基地编号: {site_costs_sorted_idx + 1}")  # +1为了显示1-10而不是0-9
    print(f"  选中基地的固定成本: {selected_sites_costs/1e9} (billion$/年)")
    print(f"  每个基地每天发射: {max_launches_per_site_per_day} 次")
    print(f"  年运输能力: {rocket_annual_capacity_kg/1e6:,.2f} 万吨")
    print(f"  所需时间: {rocket_years_needed:.2f} 年")
    print(f"  总成本: ${rocket_cost/1e9:.2f} billion美元")
    
    # 成本分解
    total_fixed = sum([b['cost_fixed'] for b in rocket_breakdown])
    total_launches = sum([b['cost_launches'] for b in rocket_breakdown])
    print(f"    - 固定运营成本: ${total_fixed/1e9:.2f} billion ({total_fixed/rocket_cost*100:.1f}%)")
    print(f"    - 发射成本: ${total_launches/1e9:.2f} billion ({total_launches/rocket_cost*100:.1f}%)")
    
    # 学习效应
    first_year_unit_cost = rocket_breakdown[0]['unit_cost']
    last_year_unit_cost = rocket_breakdown[-1]['unit_cost']
    print(f"  单位成本变化:")
    print(f"    - 初始: ${first_year_unit_cost:.2f}/kg")
    print(f"    - 最终: ${last_year_unit_cost:.2f}/kg")
    print(f"    - 降幅: {(1 - last_year_unit_cost/first_year_unit_cost)*100:.1f}%")

# 方案总结
print(f"\n【方案总结】")
total_time = balanced_result['time_years']
total_cost = balanced_result['total_cost']
avg_unit_cost = total_cost / total_materials_kg

print(f"  总完成时间: {total_time:.2f} 年 ({total_time*12:.1f} 个月)")
print(f"  总成本: ${total_cost/1e9:.2f} billion美元")
print(f"  平均单位成本: ${avg_unit_cost:.2f}/kg")

# 与极端方案对比
print(f"\n【方案对比】")

# 仅电梯方案（满负荷）
harbor_only_years = total_materials_kg / harbor_max_capacity_kg
mu_schedule_full = np.ones(int(np.ceil(harbor_only_years)))
harbor_only_cost, _, _ = calculate_harbor_cost_with_loadfactor(
    mu_schedule_full, int(np.ceil(harbor_only_years)), C1_harbor, b_harbor
)

print(f"  vs 仅电梯方案（满负荷）:")
print(f"    时间: {harbor_only_years:.2f} 年, 成本: ${harbor_only_cost/1e9:.2f} billion")
print(f"    节省时间: {harbor_only_years - total_time:.2f} 年 ({(harbor_only_years-total_time)/harbor_only_years*100:.1f}%)")
print(f"    成本增加: ${(total_cost - harbor_only_cost)/1e9:.2f} billion ({(total_cost-harbor_only_cost)/harbor_only_cost*100:.1f}%)")

# 仅火箭方案（全部基地）
rocket_only_annual_capacity = num_launch_sites * max_launches_per_site_per_day * 365 * capacity_per_rocket * 1000
rocket_only_years = total_materials_kg / rocket_only_annual_capacity
site_schedule_full = np.ones((int(np.ceil(rocket_only_years)), num_launch_sites))
launches_schedule_full = np.full((int(np.ceil(rocket_only_years)), num_launch_sites), max_launches_per_site_per_day)
rocket_only_cost, _, _ = calculate_rocket_cost_with_sites(
    site_schedule_full, launches_schedule_full, int(np.ceil(rocket_only_years)), C1_rocket, b_rocket
)

print(f"  vs 仅火箭方案（全部基地满负荷）:")
print(f"    时间: {rocket_only_years:.2f} 年, 成本: ${rocket_only_cost/1e9:.2f} billion")
print(f"    节省时间: {rocket_only_years - total_time:.2f} 年 ({(rocket_only_years-total_time)/rocket_only_years*100:.1f}%)")
print(f"    节省成本: ${(rocket_only_cost - total_cost)/1e9:.2f} billion ({(rocket_only_cost-total_cost)/rocket_only_cost*100:.1f}%)")

print("=" * 80)

In [ ]:
"""
可视化分析：改进模型的结果展示
"""

from typing import Any


fig = plt.figure(figsize=(20, 14))

# ==================== 子图1: 不同权重下的最优决策变量 ====================
ax1 = plt.subplot(3, 3, 1)
x_pos = np.arange(len(df_results))
width = 0.25

bars1 = ax1.bar(x_pos - width, df_results['alpha'] * 100, width, label='Space Elevator Ratio α (%)', color='#3498db', alpha=0.8)
bars2 = ax1.bar(x_pos, df_results['mu_harbor'] * 100, width, label='Elevator Load Rate μ (%)', color='#2ecc71', alpha=0.8)
bars3 = ax1.bar(x_pos + width, df_results['num_sites'] * 10, width, label='Rocket Bases ×10 (%)', color='#e74c3c', alpha=0.8)

ax1.set_xlabel('Strategy Weight', fontsize=11, fontweight='bold')
ax1.set_ylabel('Value (%)', fontsize=11, fontweight='bold')
ax1.set_title('Optimal Decision Variables under Different Weights', fontsize=13, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f"w1={r['w1']:.1f}" for _, r in df_results.iterrows()], rotation=0)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

# ==================== 子图2: 时间-成本帕累托前沿 ====================
ax2 = plt.subplot(3, 3, 2)

ax2.plot(df_results['time_years'], df_results['total_cost']/1e9, 'bo-', linewidth=2, markersize=10, alpha=0.6)

# 标注关键点
for _, row in df_results.iterrows():
    if row['w1'] in [0.0, 0.5, 1.0]:
        ax2.annotate(row['label'], 
                    xy=(row['time_years'], row['total_cost']/1e9),
                    xytext=(10, 10), textcoords='offset points',
                    fontsize=9, bbox=dict[str, str | float](boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.6),
                    arrowprops=dict[str, Any](arrowstyle='->', connectionstyle='arc3,rad=0'))

ax2.scatter([harbor_only_years], [harbor_only_cost/1e9], marker='s', s=200, c='green', 
           edgecolors='black', linewidth=2, label='Elevator Only', zorder=5)
ax2.scatter([rocket_only_years], [rocket_only_cost/1e9], marker='^', s=200, c='red', 
           edgecolors='black', linewidth=2, label='Rocket Only', zorder=5)

ax2.set_xlabel('Time (Years)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Cost (Billion $)', fontsize=11, fontweight='bold')
ax2.set_title('Pareto Frontier: Time-Cost Trade-off', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=9)

# ==================== 子图3: 成本构成饼图（均衡方案）====================
ax3 = plt.subplot(3, 3, 3)

if alpha_opt > 0 and (1 - alpha_opt) > 0:
    # 混合方案
    cost_components = [
        total_maint,
        total_ops,
        total_fixed,
        total_launches
    ]
    labels_pie = [
        f'Elevator Maint\n${total_maint/1e9:.2f}B',
        f'Elevator Transport\n${total_ops/1e9:.2f}B',
        f'Rocket Fixed\n${total_fixed/1e9:.2f}B',
        f'Rocket Launches\n${total_launches/1e9:.2f}B'
    ]
    colors_pie = ['#3498db', '#5dade2', '#e74c3c', '#ec7063']
elif alpha_opt == 1:
    # 仅电梯
    cost_components = [total_maint, total_ops]
    labels_pie = [f'Maint\n${total_maint/1e9:.2f}B', f'Transport\n${total_ops/1e9:.2f}B']
    colors_pie = ['#3498db', '#5dade2']
else:
    # 仅火箭
    cost_components = [total_fixed, total_launches]
    labels_pie = [f'Fixed\n${total_fixed/1e9:.2f}B', f'Launches\n${total_launches/1e9:.2f}B']
    colors_pie = ['#e74c3c', '#ec7063']

ax3.pie(cost_components, labels=labels_pie, colors=colors_pie, autopct='%1.1f%%',
        startangle=90, textprops={'fontsize': 10, 'fontweight': 'bold'})
ax3.set_title(f'Balanced Strategy Cost Composition\nTotal Cost: ${total_cost/1e9:.2f}B', fontsize=12, fontweight='bold', pad=10)

# ==================== 子图4: 运输量分配 ====================
ax4 = plt.subplot(3, 3, 4)

transport_amounts = [harbor_target_kg/1e6, rocket_target_kg/1e6]
labels_transport = [f'Space Elevator\n{harbor_target_kg/1e6:,.0f}M tons\n({alpha_opt:.1%})',
                   f'Rocket System\n{rocket_target_kg/1e6:,.0f}M tons\n({(1-alpha_opt):.1%})']
colors_transport = ['#3498db', '#e74c3c']

ax4.pie(transport_amounts, labels=labels_transport, colors=colors_transport, autopct='',
        explode=(0.05, 0.05), shadow=True, startangle=90,
        textprops={'fontsize': 11, 'fontweight': 'bold'})
ax4.set_title(f'Transport Allocation\nTotal: {total_materials:,} tons', fontsize=12, fontweight='bold', pad=10)

# ==================== 子图5: 学习曲线效应对比 ====================
ax5 = plt.subplot(3, 3, 5)

if alpha_opt > 0:
    harbor_years = [b['year'] for b in harbor_breakdown]
    harbor_unit_costs = [b['unit_cost'] for b in harbor_breakdown]
    ax5.plot(harbor_years, harbor_unit_costs, 'b-o', linewidth=2, markersize=4, label='Space Elevator', alpha=0.7)

if (1 - alpha_opt) > 0:
    rocket_years = [b['year'] for b in rocket_breakdown]
    rocket_unit_costs = [b['unit_cost'] for b in rocket_breakdown]
    ax5.plot(rocket_years, rocket_unit_costs, 'r-s', linewidth=2, markersize=4, label='Rocket System', alpha=0.7)

ax5.axhline(y=C1_harbor, color='blue', linestyle='--', linewidth=1, alpha=0.5, label=f'Elevator Initial Cost ${C1_harbor}/kg')
ax5.axhline(y=C1_rocket, color='red', linestyle='--', linewidth=1, alpha=0.5, label=f'Rocket Initial Cost ${C1_rocket}/kg')

ax5.set_xlabel('Year', fontsize=11, fontweight='bold')
ax5.set_ylabel('Unit Cost ($/kg)', fontsize=11, fontweight='bold')
ax5.set_title('Learning Curve: Unit Cost over Time', fontsize=13, fontweight='bold')
ax5.grid(True, alpha=0.3)
ax5.legend(fontsize=9)

# ==================== 子图6: 年度成本变化 ====================
ax6 = plt.subplot(3, 3, 6)

if alpha_opt > 0:
    harbor_years = [b['year'] for b in harbor_breakdown]
    harbor_year_costs = [b['year_cost']/1e9 for b in harbor_breakdown]
    ax6.bar(harbor_years, harbor_year_costs, width=0.8, label='Space Elevator', color='#3498db', alpha=0.7)

if (1 - alpha_opt) > 0:
    rocket_years = [b['year'] for b in rocket_breakdown]
    rocket_year_costs = [b['year_cost']/1e9 for b in rocket_breakdown]
    # 如果两者年份重叠，需要堆叠显示
    if alpha_opt > 0:
        ax6.bar(rocket_years, rocket_year_costs, width=0.8, label='Rocket System', 
               color='#e74c3c', alpha=0.7, bottom=harbor_year_costs[:len(rocket_years)] if len(rocket_years) <= len(harbor_years) else [0]*len(rocket_years))
    else:
        ax6.bar(rocket_years, rocket_year_costs, width=0.8, label='Rocket System', color='#e74c3c', alpha=0.7)

ax6.set_xlabel('Year', fontsize=11, fontweight='bold')
ax6.set_ylabel('Annual Cost (Billion $)', fontsize=11, fontweight='bold')
ax6.set_title('Annual Cost Distribution', fontsize=13, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3, axis='y')

# ==================== 子图7: 权重对时间和成本的影响 ====================
ax7 = plt.subplot(3, 3, 7)

ax7_twin = ax7.twinx()

line1 = ax7.plot(df_results['w1'], df_results['time_years'], 'b-o', linewidth=2.5, markersize=10, label='Time')
line2 = ax7_twin.plot(df_results['w1'], df_results['total_cost']/1e9, 'r-s', linewidth=2.5, markersize=10, label='Cost')

ax7.set_xlabel('Time Weight (w1)', fontsize=11, fontweight='bold')
ax7.set_ylabel('Time (Years)', fontsize=11, fontweight='bold', color='blue')
ax7_twin.set_ylabel('Cost (Billion $)', fontsize=11, fontweight='bold', color='red')
ax7.set_title('Weight Impact on Objectives', fontsize=13, fontweight='bold')
ax7.grid(True, alpha=0.3)
ax7.tick_params(axis='y', labelcolor='blue')
ax7_twin.tick_params(axis='y', labelcolor='red')

lines = line1 + line2
labels = [l.get_label() for l in lines]
ax7.legend(lines, labels, fontsize=10, loc='upper left')

# ==================== 子图8: 负荷率和基地数随权重变化 ====================
ax8 = plt.subplot(3, 3, 8)

ax8.plot(df_results['w1'], df_results['mu_harbor'], 'g-o', linewidth=2, markersize=8, label='Elevator Load Rate μ')
ax8_twin = ax8.twinx()
ax8_twin.plot(df_results['w1'], df_results['num_sites'], 'm-s', linewidth=2, markersize=8, label='Rocket Bases')

ax8.set_xlabel('Time Weight (w1)', fontsize=11, fontweight='bold')
ax8.set_ylabel('Elevator Load Rate', fontsize=11, fontweight='bold', color='green')
ax8_twin.set_ylabel('Rocket Bases', fontsize=11, fontweight='bold', color='magenta')
ax8.set_title('Decision Variables vs Weight', fontsize=13, fontweight='bold')
ax8.grid(True, alpha=0.3)
ax8.tick_params(axis='y', labelcolor='green')
ax8_twin.tick_params(axis='y', labelcolor='magenta')

# ==================== 子图9: 关键指标对比雷达图 ====================
ax9 = plt.subplot(3, 3, 9, projection='polar')

# 准备雷达图数据（归一化到0-1）
categories = ['Time Efficiency', 'Cost Efficiency', 'Elevator Utilization', 'Rocket Utilization', 'Overall Score']
N = len(categories)

# 计算归一化值
time_norm = 1 - (total_time - df_results['time_years'].min()) / (df_results['time_years'].max() - df_results['time_years'].min())
cost_norm = 1 - (total_cost - df_results['total_cost'].min()) / (df_results['total_cost'].max() - df_results['total_cost'].min())
harbor_util = mu_opt if alpha_opt > 0 else 0
rocket_util = sites_opt / num_launch_sites if sites_opt > 0 else 0
overall_score = 1 - (balanced_result['objective'] - df_results['objective'].min()) / (df_results['objective'].max() - df_results['objective'].min())

values = [time_norm, cost_norm, harbor_util, rocket_util, overall_score]
values += values[:1]  # 闭合

angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

ax9.plot(angles, values, 'o-', linewidth=2, color='#3498db')
ax9.fill(angles, values, alpha=0.25, color='#3498db')
ax9.set_xticks(angles[:-1])
ax9.set_xticklabels(categories, fontsize=10)
ax9.set_ylim(0, 1)
ax9.set_title('Balanced Strategy Comprehensive Evaluation', fontsize=12, fontweight='bold', pad=20)
ax9.grid(True)

plt.tight_layout()
plt.show()

print("\n图表说明：")
print("=" * 80)
print("1. 最优决策变量：展示不同权重下的太空电梯比例、负荷率和火箭基地数")
print("2. 帕累托前沿：时间-成本权衡曲线，包含极端方案对比")
print("3. 成本构成：均衡方案的详细成本分解（维护/运输/固定/发射）")
print("4. 运输量分配：太空电梯和火箭系统的物资分配比例")
print("5. 学习曲线：展示两种运输方式的单位成本随时间递减")
print("6. 年度成本分布：每年的成本支出情况")
print("7. 权重影响：时间权重如何影响总时间和总成本")
print("8. 决策变量变化：负荷率和基地数随权重的变化趋势")
print("9. 综合评价雷达图：多维度评估均衡方案的性能")
print("=" * 80)

In [ ]:
"""
敏感性分析：关键参数对最优解的影响
"""

print("\n" + "=" * 80)
print("敏感性分析：关键参数变化的影响".center(80))
print("=" * 80)

# 固定使用均衡权重
w1_test, w2_test = 0.5, 0.5

# ==================== 分析1: 学习率参数的影响 ====================
print("\n【分析1：学习率参数的影响】")
print("-" * 80)

b_harbor_test_values = [0.05, 0.10, 0.15, 0.20, 0.25]
b_rocket_test_values = [0.10, 0.15, 0.20, 0.25, 0.30]

print(f"\n1.1 太空电梯学习率 b_harbor 的影响（火箭固定 b={b_rocket}）:")
print(f"{'b_harbor':<12} {'最优α':<10} {'最优μ':<10} {'基地数':<10} {'时间(年)':<12} {'成本(billion$)':<12}")
print("-" * 80)

for b_h_test in b_harbor_test_values:
    # 使用原始参数，只修改b_harbor
    best_obj = float('inf')
    best_sol = None
    
    for alpha in np.linspace(0, 1, 11):
        for mu_h in [0.5, 0.7, 0.9, 1.0]:
            for n_sites in range(0, num_launch_sites + 1):
                if alpha == 0 and mu_h > 0: continue
                if alpha == 1 and n_sites > 0: continue
                if alpha > 0 and mu_h == 0: continue
                if alpha < 1 and n_sites == 0: continue
                
                # 临时修改全局变量（不好的做法，但为了简化）
                # 实际应该把b作为参数传入函数
                # 这里用一个辅助函数
                def eval_with_custom_b(alpha, mu_h, n_sites, b_h_custom):
                    harbor_target = total_materials_kg * alpha
                    rocket_target = total_materials_kg * (1 - alpha)
                    
                    if harbor_target > 0:
                        harbor_annual_cap = mu_h * harbor_max_capacity_kg
                        harbor_yrs = harbor_target / harbor_annual_cap
                        harbor_total_yrs = int(np.ceil(harbor_yrs))
                        mu_sched = np.full(harbor_total_yrs, mu_h)
                        if harbor_yrs % 1 > 0:
                            mu_sched[-1] = mu_h * (harbor_yrs % 1)
                        h_cost, _, _ = calculate_harbor_cost_with_loadfactor(
                            mu_sched, harbor_total_yrs, C1_harbor, b_h_custom
                        )
                        h_time = harbor_yrs
                    else:
                        h_cost, h_time = 0, 0
                    
                    if rocket_target > 0 and n_sites > 0:
                        sites_idx = np.argsort(F_sites)[:int(n_sites)]
                        r_annual_cap = n_sites * max_launches_per_site_per_day * 365 * capacity_per_rocket * 1000
                        r_yrs = rocket_target / r_annual_cap
                        r_total_yrs = int(np.ceil(r_yrs))
                        
                        site_sched = np.zeros((r_total_yrs, num_launch_sites))
                        launches_sched = np.zeros((r_total_yrs, num_launch_sites))
                        for yr_idx in range(r_total_yrs):
                            for s_idx in sites_idx:
                                site_sched[yr_idx, s_idx] = 1
                                if yr_idx == r_total_yrs - 1 and r_yrs % 1 > 0:
                                    launches_sched[yr_idx, s_idx] = max_launches_per_site_per_day * (r_yrs % 1)
                                else:
                                    launches_sched[yr_idx, s_idx] = max_launches_per_site_per_day
                        
                        r_cost, _, _ = calculate_rocket_cost_with_sites(
                            site_sched, launches_sched, r_total_yrs, C1_rocket, b_rocket
                        )
                        r_time = r_yrs
                    else:
                        r_cost, r_time = 0, 0
                    
                    total_t = max(h_time, r_time) if (h_time > 0 or r_time > 0) else 0
                    total_c = h_cost + r_cost
                    
                    if total_t > 0 and total_t <= T_max:
                        obj = w1_test * (total_t / T_max) + w2_test * (total_c / (C1_harbor * total_materials_kg))
                        return True, total_t, total_c, obj
                    else:
                        return False, 0, 0, 1e10
                
                is_feas, t, c, obj = eval_with_custom_b(alpha, mu_h, n_sites, b_h_test)
                
                if is_feas and obj < best_obj:
                    best_obj = obj
                    best_sol = {'alpha': alpha, 'mu': mu_h, 'sites': n_sites, 'time': t, 'cost': c}
    
    if best_sol:
        print(f"{b_h_test:<12.2f} {best_sol['alpha']:<10.2%} {best_sol['mu']:<10.2%} "
              f"{best_sol['sites']:<10.0f} {best_sol['time']:<12.2f} {best_sol['cost']/1e9:<12.2f}")

print(f"\n1.2 火箭学习率 b_rocket 的影响（电梯固定 b={b_harbor}）:")
print(f"{'b_rocket':<12} {'最优α':<10} {'最优μ':<10} {'基地数':<10} {'时间(年)':<12} {'成本(billion$)':<12}")
print("-" * 80)

for b_r_test in b_rocket_test_values:
    best_obj = float('inf')
    best_sol = None
    
    for alpha in np.linspace(0, 1, 11):
        for mu_h in [0.5, 0.7, 0.9, 1.0]:
            for n_sites in range(0, num_launch_sites + 1):
                if alpha == 0 and mu_h > 0: continue
                if alpha == 1 and n_sites > 0: continue
                if alpha > 0 and mu_h == 0: continue
                if alpha < 1 and n_sites == 0: continue
                
                def eval_with_custom_b_rocket(alpha, mu_h, n_sites, b_r_custom):
                    harbor_target = total_materials_kg * alpha
                    rocket_target = total_materials_kg * (1 - alpha)
                    
                    if harbor_target > 0:
                        harbor_annual_cap = mu_h * harbor_max_capacity_kg
                        harbor_yrs = harbor_target / harbor_annual_cap
                        harbor_total_yrs = int(np.ceil(harbor_yrs))
                        mu_sched = np.full(harbor_total_yrs, mu_h)
                        if harbor_yrs % 1 > 0:
                            mu_sched[-1] = mu_h * (harbor_yrs % 1)
                        h_cost, _, _ = calculate_harbor_cost_with_loadfactor(
                            mu_sched, harbor_total_yrs, C1_harbor, b_harbor
                        )
                        h_time = harbor_yrs
                    else:
                        h_cost, h_time = 0, 0
                    
                    if rocket_target > 0 and n_sites > 0:
                        sites_idx = np.argsort(F_sites)[:int(n_sites)]
                        r_annual_cap = n_sites * max_launches_per_site_per_day * 365 * capacity_per_rocket * 1000
                        r_yrs = rocket_target / r_annual_cap
                        r_total_yrs = int(np.ceil(r_yrs))
                        
                        site_sched = np.zeros((r_total_yrs, num_launch_sites))
                        launches_sched = np.zeros((r_total_yrs, num_launch_sites))
                        for yr_idx in range(r_total_yrs):
                            for s_idx in sites_idx:
                                site_sched[yr_idx, s_idx] = 1
                                if yr_idx == r_total_yrs - 1 and r_yrs % 1 > 0:
                                    launches_sched[yr_idx, s_idx] = max_launches_per_site_per_day * (r_yrs % 1)
                                else:
                                    launches_sched[yr_idx, s_idx] = max_launches_per_site_per_day
                        
                        r_cost, _, _ = calculate_rocket_cost_with_sites(
                            site_sched, launches_sched, r_total_yrs, C1_rocket, b_r_custom
                        )
                        r_time = r_yrs
                    else:
                        r_cost, r_time = 0, 0
                    
                    total_t = max(h_time, r_time) if (h_time > 0 or r_time > 0) else 0
                    total_c = h_cost + r_cost
                    
                    if total_t > 0 and total_t <= T_max:
                        obj = w1_test * (total_t / T_max) + w2_test * (total_c / (C1_harbor * total_materials_kg))
                        return True, total_t, total_c, obj
                    else:
                        return False, 0, 0, 1e10
                
                is_feas, t, c, obj = eval_with_custom_b_rocket(alpha, mu_h, n_sites, b_r_test)
                
                if is_feas and obj < best_obj:
                    best_obj = obj
                    best_sol = {'alpha': alpha, 'mu': mu_h, 'sites': n_sites, 'time': t, 'cost': c}
    
    if best_sol:
        print(f"{b_r_test:<12.2f} {best_sol['alpha']:<10.2%} {best_sol['mu']:<10.2%} "
              f"{best_sol['sites']:<10.0f} {best_sol['time']:<12.2f} {best_sol['cost']/1e9:<12.2f}")
print("=" * 80)